In [ ]:
import json
import copy
import random 
import re
from pathlib import Path


import pandas as pd
import numpy as np
import requests 

project_root = Path.cwd().parent
data_dir = project_root / "data"
results_dir = project_root / "results"

scenario_path = data_dir / "pilot_scenarios.json"

In [ ]:
# load final results (1 sample)

df = pd.read_csv(results_dir / "qwen7b_randomised_results.csv")

# define target responses 

def get_target_option(row):
    if row["condition"] == "listener":
        return "target_false_inference"
    elif row["condition"] == "speaker_neutral":
        return "fair_response"
    elif row["condition"] == "speaker_goal":
        return "true_but_misleading"
    else:
        return np.nan


df["target_option_id"] = df.apply(get_target_option, axis=1)

df["target_selected"] = (
    df["chosen_option_id"] == df["target_option_id"]
)

df["step1_target_selected"] = (
    (df["condition"] == "listener_speaker") &
    (df["step1_option_id"] == "target_false_inference")
)

df["step2_target_selected"] = (
    (df["condition"] == "listener_speaker") &
    (df["step2_option_id"] == "true_but_misleading")
)


# calculate target response rates

# for single choice conditions: 

summary_counts = (
    df[df["condition"] != "listener_speaker"]
    .groupby("condition")["target_selected"]
    .agg(["sum", "count", "mean"])
    .reset_index()
)

summary_counts["target_selected_percent"] = summary_counts["mean"] * 100

# listener-speaker step 1:

ls_df = df[df["condition"] == "listener_speaker"]

ls_step1 = pd.DataFrame([{
    "condition": "listener_speaker_step1",
    "sum": ls_df["step1_target_selected"].sum(),
    "count": len(ls_df),
    "mean": ls_df["step1_target_selected"].mean(),
    "target_selected_percent": ls_df["step1_target_selected"].mean() * 100,
}])

# listener-speaker step 2:

ls_step2 = pd.DataFrame([{
    "condition": "listener_speaker_step2",
    "sum": ls_df["step2_target_selected"].sum(),
    "count": len(ls_df),
    "mean": ls_df["step2_target_selected"].mean(),
    "target_selected_percent": ls_df["step2_target_selected"].mean() * 100,
}])

target_summary = pd.concat(
    [summary_counts, ls_step1, ls_step2],
    ignore_index=True
)


# calculate most selected response 

# for single choice conditions: 

modal_main = (
    df[df["condition"] != "listener_speaker"]
    .groupby("condition")["chosen_option_id"]
    .value_counts(normalize=True)
    .mul(100)
    .reset_index(name="most_selected_percent")
)

modal_main = (
    modal_main
    .sort_values(["condition", "most_selected_percent"], ascending=[True, False])
    .groupby("condition")
    .head(1)
    .rename(columns={"chosen_option_id": "most_selected_response"})
)

# listener-speaker step 1:

ls_step1_modal = (
    ls_df["step1_option_id"]
    .value_counts(normalize=True)
    .mul(100)
    .reset_index(name="most_selected_percent")
    .rename(columns={"step1_option_id": "most_selected_response"})
)

ls_step1_modal["condition"] = "listener_speaker_step1"
ls_step1_modal = ls_step1_modal.head(1)

# listener-speaker step 2:

ls_step2_modal = (
    ls_df["step2_option_id"]
    .value_counts(normalize=True)
    .mul(100)
    .reset_index(name="most_selected_percent")
    .rename(columns={"step2_option_id": "most_selected_response"})
)

ls_step2_modal["condition"] = "listener_speaker_step2"
ls_step2_modal = ls_step2_modal.head(1)

modal_summary = pd.concat(
    [modal_main, ls_step1_modal, ls_step2_modal],
    ignore_index=True
)

# merge target rates + modal responses

final_summary = target_summary.merge(
    modal_summary,
    on="condition",
    how="left"
)

# clean columns: 

final_summary = final_summary[[
    "condition",
    "sum",
    "count",
    "target_selected_percent",
    "most_selected_response",
    "most_selected_percent",
]]

# order conditions: 

condition_order = [
    "listener",
    "speaker_neutral",
    "speaker_goal",
    "listener_speaker_step1",
    "listener_speaker_step2",
]

final_summary["condition"] = pd.Categorical(
    final_summary["condition"],
    categories=condition_order,
    ordered=True
)

final_summary = final_summary.sort_values("condition").reset_index(drop=True)

print(final_summary)

In [ ]:
# load 5 sample results

df = pd.read_csv(results_dir / "qwen7b_5sample_results.csv")

# define target responses: 

def get_target_option(row):
    if row["condition"] == "listener":
        return "target_false_inference"
    elif row["condition"] == "speaker_neutral":
        return "fair_response"
    elif row["condition"] == "speaker_goal":
        return "true_but_misleading"
    else:
        return np.nan


df["target_option_id"] = df.apply(get_target_option, axis=1)

df["target_selected"] = (
    df["chosen_option_id"] == df["target_option_id"]
)

df["step1_target_selected"] = (
    (df["condition"] == "listener_speaker") &
    (df["step1_option_id"] == "target_false_inference")
)

df["step2_target_selected"] = (
    (df["condition"] == "listener_speaker") &
    (df["step2_option_id"] == "true_but_misleading")
)

# calculate target response rates

# for single choice conditions:

summary_counts = (
    df[df["condition"] != "listener_speaker"]
    .groupby("condition")["target_selected"]
    .agg(["sum", "count", "mean"])
    .reset_index()
)

summary_counts["target_selected_percent"] = summary_counts["mean"] * 100

# listener-speaker step 1:

ls_df = df[df["condition"] == "listener_speaker"]

ls_step1 = pd.DataFrame([{
    "condition": "listener_speaker_step1",
    "sum": ls_df["step1_target_selected"].sum(),
    "count": len(ls_df),
    "mean": ls_df["step1_target_selected"].mean(),
    "target_selected_percent": ls_df["step1_target_selected"].mean() * 100,
}])

# listener-speaker step 2:

ls_step2 = pd.DataFrame([{
    "condition": "listener_speaker_step2",
    "sum": ls_df["step2_target_selected"].sum(),
    "count": len(ls_df),
    "mean": ls_df["step2_target_selected"].mean(),
    "target_selected_percent": ls_df["step2_target_selected"].mean() * 100,
}])

target_summary = pd.concat(
    [summary_counts, ls_step1, ls_step2],
    ignore_index=True
)

# most selected response per condition

# single choice conditions: 

modal_main = (
    df[df["condition"] != "listener_speaker"]
    .groupby("condition")["chosen_option_id"]
    .value_counts(normalize=True)
    .mul(100)
    .reset_index(name="most_selected_percent")
)

modal_main = (
    modal_main
    .sort_values(["condition", "most_selected_percent"], ascending=[True, False])
    .groupby("condition")
    .head(1)
    .rename(columns={"chosen_option_id": "most_selected_response"})
)

# listener-speaker step 1: 

ls_step1_modal = (
    ls_df["step1_option_id"]
    .value_counts(normalize=True)
    .mul(100)
    .reset_index(name="most_selected_percent")
    .rename(columns={"step1_option_id": "most_selected_response"})
)

ls_step1_modal["condition"] = "listener_speaker_step1"
ls_step1_modal = ls_step1_modal.head(1)

# listener-speaker step 2:

ls_step2_modal = (
    ls_df["step2_option_id"]
    .value_counts(normalize=True)
    .mul(100)
    .reset_index(name="most_selected_percent")
    .rename(columns={"step2_option_id": "most_selected_response"})
)

ls_step2_modal["condition"] = "listener_speaker_step2"
ls_step2_modal = ls_step2_modal.head(1)

modal_summary = pd.concat(
    [modal_main, ls_step1_modal, ls_step2_modal],
    ignore_index=True
)

# merge target rates + modal responses 


final_summary = target_summary.merge(
    modal_summary,
    on="condition",
    how="left"
)

# clean columns

final_summary = final_summary[[
    "condition",
    "sum",
    "count",
    "target_selected_percent",
    "most_selected_response",
    "most_selected_percent",
]]

# order conditions

condition_order = [
    "listener",
    "speaker_neutral",
    "speaker_goal",
    "listener_speaker_step1",
    "listener_speaker_step2",
]

final_summary["condition"] = pd.Categorical(
    final_summary["condition"],
    categories=condition_order,
    ordered=True
)

final_summary = final_summary.sort_values("condition").reset_index(drop=True)

print(final_summary)

In [ ]:
# distortion manipulation

# single choice conditions: 

by_distortion = (
    df[df["condition"] != "listener_speaker"]
    .groupby(["condition", "distortion_level"])["target_selected"]
    .agg(["sum", "count", "mean"])
    .reset_index()
)

by_distortion["percent"] = by_distortion["mean"] * 100
print(by_distortion)


# listener-speaker: 

ls_by_distortion = (
    df[df["condition"] == "listener_speaker"]
    .groupby("distortion_level")[["step1_target_selected", "step2_target_selected"]]
    .mean()
    .mul(100)
    .reset_index()
)

print(ls_by_distortion)

In [ ]:
speaker_neutral_tbm = (
    df[df["condition"] == "speaker_neutral"]["chosen_option_id"]
    .eq("true_but_misleading")
    .mean()
    * 100
)

speaker_neutral_tbm_n = (
    df[df["condition"] == "speaker_neutral"]["chosen_option_id"]
    .eq("true_but_misleading")
    .sum()
)

print(speaker_neutral_tbm_n, speaker_neutral_tbm)

In [ ]:
# speaker conditions comparison - neutral vs goal-directed

df = pd.read_csv(results_dir / "qwen7b_randomised_results.csv")

speaker_conditions = df[df["condition"].isin(["speaker_neutral", "speaker_goal"])].copy()

tbm_compare = (
    speaker_conditions
    .assign(tbm_selected=speaker_conditions["chosen_option_id"].eq("true_but_misleading"))
    .groupby("condition")["tbm_selected"]
    .agg(["sum", "count", "mean"])
    .reset_index()
)

tbm_compare["percent"] = tbm_compare["mean"] * 100

print(tbm_compare)

In [ ]:
# speaker conditions comparison - neutral vs goal-directed

df = pd.read_csv("qwen7b_5sample_results.csv")

speaker_conditions = df[df["condition"].isin(["speaker_neutral", "speaker_goal"])].copy()

tbm_compare = (
    speaker_conditions
    .assign(tbm_selected=speaker_conditions["chosen_option_id"].eq("true_but_misleading"))
    .groupby("condition")["tbm_selected"]
    .agg(["sum", "count", "mean"])
    .reset_index()
)

tbm_compare["percent"] = tbm_compare["mean"] * 100

print(tbm_compare)